In [ ]:
# ============================================================
# Cell 0: 硬件自检 —— 概念章，CPU 即可
# ============================================================
# 本章用 gymnasium 自带的 CartPole-v1 环境 + 一个 4→32→2 的策略网络
# 跑 REINFORCE，CPU 几十秒训到收敛，无需 GPU。
import torch

print('PyTorch version:', torch.__version__)
print('CUDA available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device         :', torch.cuda.get_device_name(0))
else:
    print('未检测到 GPU —— 本章 CPU 完全够用，继续执行即可。')


In [ ]:
%%capture
# ============================================================
# Cell 1: 安装/升级依赖库
# ============================================================
# 重要：%%capture 必须是 cell 第一行。
# gymnasium：经典 RL 环境集合（CartPole / Pendulum / LunarLander 等）
#            是 OpenAI gym 的官方延续维护版，API 几乎一致。
# matplotlib：画训练回报曲线
!pip install -q -U gymnasium matplotlib


In [ ]:
# ============================================================
# Cell 2: 看一遍 CartPole-v1 环境长什么样
# ============================================================
# CartPole：小车上立一根杆，每一步可以推车向左 / 向右；杆倒下或越界 episode 结束。
# 状态 s ∈ R^4：[小车位置, 小车速度, 杆角度, 杆角速度]
# 动作 a ∈ {0, 1}：0=左推，1=右推
# 即时奖励 r：每多撑一步 +1（活得越久回报越大）
# 终止条件：杆角度太大 / 小车出界 / 步数达到上限 500（v1 版本）
import gymnasium as gym

env = gym.make('CartPole-v1')
print('observation_space:', env.observation_space)
print('action_space     :', env.action_space)

# 跑一条随机策略的 episode 看看长度
s, _ = env.reset(seed=0)
total_r = 0
steps = 0
while True:
    a = env.action_space.sample()                # 完全随机 —— 不学习
    s, r, terminated, truncated, _ = env.step(a)
    total_r += r
    steps += 1
    if terminated or truncated:
        break

print(f'\n随机策略一条 episode：steps = {steps}, total_reward = {total_r}')
print('（随机策略基本撑不过 20 ~ 30 步——杆几秒就倒）')
env.close()


In [ ]:
# ============================================================
# Cell 3: 随机策略 baseline —— 多跑几条算平均回报
# ============================================================
# 训练前先跑一次 baseline，方便后面对比 REINFORCE 训练后的提升量。
import gymnasium as gym
import numpy as np

env = gym.make('CartPole-v1')

def run_random(n_episodes=50):
    returns = []
    for ep in range(n_episodes):
        s, _ = env.reset(seed=ep)
        total = 0
        while True:
            a = env.action_space.sample()
            s, r, terminated, truncated, _ = env.step(a)
            total += r
            if terminated or truncated:
                break
        returns.append(total)
    return returns

returns_random = run_random()
print(f'随机策略 50 条 episode：平均 return = {np.mean(returns_random):.1f}, '
      f'max = {np.max(returns_random):.0f}, min = {np.min(returns_random):.0f}')
print('（CartPole-v1 满分 500；随机策略平均通常在 20 上下）')
env.close()


In [ ]:
# ============================================================
# Cell 4: 策略网络 + REINFORCE 训练循环
# ============================================================
# 策略：4 → 32 → 2 的 MLP；输出 raw logits，softmax 之后是 Categorical 分布。
# 算法：朴素 REINFORCE（无 baseline，配合下面 Cell 5 对比效果）。
import torch
import torch.nn as nn
import torch.nn.functional as F
import gymnasium as gym
import numpy as np

class PolicyNet(nn.Module):
    def __init__(self, obs_dim=4, n_actions=2, hidden=32):
        super().__init__()
        self.fc1 = nn.Linear(obs_dim, hidden)
        self.fc2 = nn.Linear(hidden, n_actions)
    def forward(self, s):
        # 输出 raw logits —— 不要在末尾加 softmax；让 Categorical 内部去做
        return self.fc2(F.relu(self.fc1(s)))

def discount_returns(rewards, gamma):
    """从后往前算 G_t = r_t + γ r_{t+1} + γ^2 r_{t+2} + ...  (P05 §2.3 公式)"""
    G = 0.0
    out = []
    for r in reversed(rewards):
        G = r + gamma * G
        out.insert(0, G)
    return out

def train_reinforce(use_baseline=False, n_episodes=600, lr=1e-2, gamma=0.99, seed=0):
    torch.manual_seed(seed)
    np.random.seed(seed)
    env = gym.make('CartPole-v1')
    policy = PolicyNet()
    optim  = torch.optim.AdamW(policy.parameters(), lr=lr)

    history = []
    for ep in range(n_episodes):
        # ---- ① rollout：用当前策略走完一条 episode ----
        s, _ = env.reset(seed=seed * 1000 + ep)
        states, actions, rewards = [], [], []
        while True:
            s_t = torch.as_tensor(s, dtype=torch.float32)
            logits = policy(s_t)                                  # (2,)
            dist   = torch.distributions.Categorical(logits=logits)
            a      = dist.sample().item()
            s2, r, terminated, truncated, _ = env.step(a)
            states.append(s); actions.append(a); rewards.append(r)
            s = s2
            if terminated or truncated:
                break

        # ---- ② 算每步的 return G_t ----
        returns = torch.tensor(discount_returns(rewards, gamma), dtype=torch.float32)
        # ★ baseline：把 returns 减去均值除以标准差 —— 简单粗暴但极有效
        if use_baseline:
            returns = (returns - returns.mean()) / (returns.std() + 1e-8)

        # ---- ③ 重新 forward 整条 trajectory，拿到 log π(a|s) ----
        states_t  = torch.as_tensor(np.array(states), dtype=torch.float32)
        actions_t = torch.as_tensor(actions, dtype=torch.long)
        logits_t  = policy(states_t)                               # (T, 2)
        log_probs = torch.distributions.Categorical(logits=logits_t).log_prob(actions_t)  # (T,)

        # ---- ④ REINFORCE loss = -E[ log π(a|s) · G ]  ----
        # 注意取负号：PyTorch 优化器是做 minimization 的；要"最大化 J"就把 loss 写成 -J
        loss = -(log_probs * returns).mean()
        optim.zero_grad()
        loss.backward()
        optim.step()

        history.append(sum(rewards))                                # 真实 episode return（不是归一化后的）
        if (ep + 1) % 50 == 0:
            avg = np.mean(history[-50:])
            print(f'  episode {ep+1:4d}  avg(last 50) = {avg:6.1f}')
    env.close()
    return history

print('===== 训练 1：朴素 REINFORCE（no baseline）=====')
hist_no = train_reinforce(use_baseline=False)


In [ ]:
# ============================================================
# Cell 5: 加 baseline 重训一次 —— 看方差降低 / 收敛加速
# ============================================================
# baseline 的实现就是上面 Cell 4 那一行 (returns - mean) / std。
# 数学上期望不变但方差大幅下降——P05 §7 的实证。
print('===== 训练 2：REINFORCE + 简单 baseline =====')
hist_bl = train_reinforce(use_baseline=True)


In [ ]:
# ============================================================
# Cell 6: 画两次训练的曲线 —— 对比 baseline 的效果
# ============================================================
# CartPole-v1 的"满分"是 500；折线波动越小说明梯度估计方差越低。
import numpy as np
import matplotlib.pyplot as plt

def smooth(xs, w=20):
    """滑动平均，看趋势更直观；w 是窗口大小。"""
    xs = np.asarray(xs, dtype=float)
    if len(xs) < w:
        return xs
    kernel = np.ones(w) / w
    return np.convolve(xs, kernel, mode='valid')

plt.figure(figsize=(9, 4))
plt.plot(hist_no, alpha=0.25, color='C0', label='no baseline (raw)')
plt.plot(hist_bl, alpha=0.25, color='C1', label='+ baseline (raw)')
plt.plot(np.arange(len(smooth(hist_no))) + 10, smooth(hist_no), color='C0', linewidth=2, label='no baseline (smoothed)')
plt.plot(np.arange(len(smooth(hist_bl))) + 10, smooth(hist_bl), color='C1', linewidth=2, label='+ baseline (smoothed)')
plt.axhline(500, color='gray', linestyle='--', label='env max return = 500')
plt.xlabel('episode'); plt.ylabel('return')
plt.title('REINFORCE on CartPole-v1: effect of baseline on training stability')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

print(f'no baseline   末 50 集平均 return = {np.mean(hist_no[-50:]):.1f}')
print(f'+ baseline    末 50 集平均 return = {np.mean(hist_bl[-50:]):.1f}')
print('观察：加了 baseline 后曲线明显更平滑、更早接近 500。')
print('     这是 P05 §7 "降方差"那一节在数据上的实证。')


In [ ]:
# ============================================================
# Cell 7: 评测训练好的策略 —— 跑 50 条 episode 看回报
# ============================================================
# 训练时是采样动作（带探索）；评测时用 argmax（贪心）即可。
import torch
import torch.nn.functional as F
import gymnasium as gym
import numpy as np

# 重新用 baseline 配置训一遍并保留模型，避免与上面变量混淆
torch.manual_seed(123)
env = gym.make('CartPole-v1')
policy = PolicyNet()
optim  = torch.optim.AdamW(policy.parameters(), lr=1e-2)

# 复用 Cell 4 的训练循环 —— 这里直接 inline 一遍，避免依赖隐藏状态
for ep in range(600):
    s, _ = env.reset(seed=ep)
    states, actions, rewards = [], [], []
    while True:
        s_t = torch.as_tensor(s, dtype=torch.float32)
        logits = policy(s_t)
        a = torch.distributions.Categorical(logits=logits).sample().item()
        s2, r, terminated, truncated, _ = env.step(a)
        states.append(s); actions.append(a); rewards.append(r)
        s = s2
        if terminated or truncated:
            break
    G = 0.0; returns = []
    for r in reversed(rewards):
        G = r + 0.99 * G
        returns.insert(0, G)
    returns = torch.tensor(returns, dtype=torch.float32)
    returns = (returns - returns.mean()) / (returns.std() + 1e-8)
    states_t  = torch.as_tensor(np.array(states), dtype=torch.float32)
    actions_t = torch.as_tensor(actions, dtype=torch.long)
    logits_t  = policy(states_t)
    log_probs = torch.distributions.Categorical(logits=logits_t).log_prob(actions_t)
    loss = -(log_probs * returns).mean()
    optim.zero_grad(); loss.backward(); optim.step()
env.close()

# ---- 评测 ----
env = gym.make('CartPole-v1')
returns_eval = []
for ep in range(50):
    s, _ = env.reset(seed=10000 + ep)
    total = 0
    while True:
        with torch.no_grad():
            logits = policy(torch.as_tensor(s, dtype=torch.float32))
            a = int(torch.argmax(logits).item())                    # greedy
        s, r, terminated, truncated, _ = env.step(a)
        total += r
        if terminated or truncated:
            break
    returns_eval.append(total)
env.close()

print(f'训练后贪心策略 50 条 episode：avg = {np.mean(returns_eval):.1f}, '
      f'max = {np.max(returns_eval):.0f}, min = {np.min(returns_eval):.0f}')
print(f'对比：随机策略 baseline 平均 ≈ 20，训练后通常稳定接近 500（满分）。')


In [ ]:
# ============================================================
# Cell 8: 把"RL 到 LLM"的映射用最小代码实证一遍
# ============================================================
# 演示一个简化到极致的"LLM 风格" REINFORCE 闭环：
#   - 状态：迄今生成的 token 序列
#   - 动作：下一个 token（词表大小 V=5 的迷你"语言模型"）
#   - 奖励：序列长度达到 8 时整体打分（"正确序列" +1，否则 0）
#   - 模型：给定前缀，输出下一个 token 的 logits（一个简单 MLP）
#
# 这和真正的 LLM RLHF 训练循环结构完全一致——只是把"生成 token"换成"挑动作"，
# 把"RM 给打分"简化成"序列等于目标就 +1"。
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

V = 5                                  # 词表大小
T = 8                                  # 序列长度（episode 长度）
TARGET = [1, 2, 3, 4, 1, 2, 3, 4]      # 目标"正确序列"

class TinyLM(nn.Module):
    """给定迄今为止的 token 序列，输出下一个 token 的 logits。"""
    def __init__(self, V=V, T=T, hidden=32):
        super().__init__()
        # 把 (历史 token id 序列) 用 embedding 求和当输入（极简版"上下文表示"）
        self.emb = nn.Embedding(V + 1, hidden)        # +1 给 BOS
        self.head = nn.Linear(hidden, V)
    def forward(self, prefix_ids):
        # prefix_ids: (B, t) long；输出 (B, V) raw logits
        x = self.emb(prefix_ids).mean(dim=1)
        return self.head(x)

def rollout(model, n=32):
    """一次性 rollout n 条序列。返回每条的 token 序列、log_prob 之和、奖励。"""
    BOS = V                                            # 特殊 BOS id
    prefix = torch.full((n, 1), BOS, dtype=torch.long)
    log_probs = torch.zeros(n)
    tokens = []
    for t in range(T):
        logits = model(prefix)                         # (n, V)
        dist = torch.distributions.Categorical(logits=logits)
        a = dist.sample()                              # (n,)
        log_probs = log_probs + dist.log_prob(a)       # 累加每步 log π(a|s)
        prefix = torch.cat([prefix, a.unsqueeze(1)], dim=1)
        tokens.append(a)
    seqs = torch.stack(tokens, dim=1)                  # (n, T)
    # 奖励：序列等于 TARGET 拿 +1，否则 0（极稀疏奖励，演示用）
    target = torch.tensor(TARGET, dtype=torch.long)
    rewards = (seqs == target).all(dim=1).float()      # (n,)
    return seqs, log_probs, rewards

torch.manual_seed(0)
model = TinyLM()
optim = torch.optim.AdamW(model.parameters(), lr=1e-2)

batch_size = 64
hits = []
for step in range(300):
    seqs, log_probs, rewards = rollout(model, n=batch_size)
    # baseline：减去 batch 内平均奖励 —— 同 §7 的简单方案
    advantage = rewards - rewards.mean()
    # 策略梯度 loss：-E[ log π · A ] —— 这就是"LLM RL 训练"的核心一行
    loss = -(log_probs * advantage).mean()
    optim.zero_grad(); loss.backward(); optim.step()

    hits.append(rewards.mean().item())
    if (step + 1) % 30 == 0:
        print(f'step {step+1:3d}  hit rate = {hits[-1]:.3f}  '
              f'(目标序列在 batch 中出现的比例)')

import matplotlib.pyplot as plt
plt.figure(figsize=(7, 3.5))
plt.plot(hits)
plt.xlabel('step'); plt.ylabel('hit rate (= avg reward)')
plt.title('Mini LLM-RL loop: policy gradient raises probability of target sequence')
plt.grid(alpha=0.3)
plt.show()

print('\n回顾：上面这套循环和真正的 RLHF 几乎一字不差——区别只是：')
print('  - 模型从 TinyLM 换成 Qwen / LLaMA')
print('  - 奖励从"序列等于 TARGET"换成"奖励模型 RM(prompt, response)"')
print('  - 还要加 KL 约束 β · D_KL(π_θ ∥ π_ref) 防止跑偏（P03 §6 的 KL）')
print('  - PPO / GRPO 还会加 importance sampling + clip 让一个 batch 数据能用多步')
